In [ ]:
#install packages

#Geopandas is for creating map plots out of pandas data frames
!pip install geopandas

#Contextily is to add base maps to map plots
!pip install contextily

In [ ]:
#import packages
import geopandas as gpd

import contextily as ctx

import matplotlib.pyplot as plt

import pandas as pd

import csv

In [ ]:
#first file to be uploaded
#This file is from the 

infile = 'ACSDT1Y2023.B07011-Data.csv'

#creates empty list
GeoMobilityIncomeList = []

#open file to read
with open(infile, 'r') as csvfile:
  CensusReader = csv.reader(csvfile,  dialect='excel', delimiter=',')


#begin for loop to read in data into list
  for line in CensusReader:
    if line[0] == '' or line[2].startswith('null'):
      continue
    else:
      try:
        #creates the dictionary and entries
        GeoMobilityIncome = {}

        GeoMobilityIncome['geo id'] = line[0]
        GeoMobilityIncome['county name'] = line[1]
        GeoMobilityIncome['median income past 12 months'] = line[2]
        GeoMobilityIncome['margin error past 12 months'] = line[3]
        GeoMobilityIncome['median income past 12 months same house 1 year ago'] = line[4]
        GeoMobilityIncome['margin error past 12 months same house 1 year ago'] = line[5]
        GeoMobilityIncome['median income past 12 months moved within same county'] = line[6]
        GeoMobilityIncome['margin error past 12 months moved within same county'] = line[7]
        GeoMobilityIncome['median income past 12 months moved from different county same state'] = line[8]
        GeoMobilityIncome['margin error past 12 months moved from different county same state'] = line[9]
        GeoMobilityIncome['median income past 12 months moved from different state'] = line[10]
        GeoMobilityIncome['margin error past 12 months moved from different state'] = line[11]
        GeoMobilityIncome['median income past 12 months moved from abroad'] = line[12]
        GeoMobilityIncome['margin of error past 12 months moved from abroad'] = line[13]


        #adds the dictionary entries into an account list
        GeoMobilityIncomeList.append(GeoMobilityIncome)

      except IndexError:
        print('Error: ', line)

csvfile.close()
#prints number of read entries
print('Read', len(GeoMobilityIncomeList), 'counties')

In [ ]:
#remove first 2 rows
GeoMobilityIncomeList = GeoMobilityIncomeList[2:]

print(GeoMobilityIncomeList)

In [ ]:
#create pandas DF
GeoMobilityIncomeDF = pd.DataFrame(GeoMobilityIncomeList)

GeoMobilityIncomeDF

In [ ]:
#clean up data, replace nulls, blanks and other symbols with 0

GeoMobilityIncomeDF = GeoMobilityIncomeDF.fillna(0) # Replace NaN with 0
GeoMobilityIncomeDF = GeoMobilityIncomeDF.replace('-', 0) # Replace - with 0
GeoMobilityIncomeDF = GeoMobilityIncomeDF.replace('**', 0) # Replace ** with 0
GeoMobilityIncomeDF = GeoMobilityIncomeDF.replace('***', 0) # Replace *** with 0

GeoMobilityIncomeDF

In [ ]:
#changing the type of all columns to numeric

#had to change to str, remove commas and change to numeric
GeoMobilityIncomeDF['median income past 12 months'] = pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months'].astype(str).str.replace(',', ''))
#had to change to str, remove commas and change to numeric
GeoMobilityIncomeDF['median income past 12 months same house 1 year ago'] = pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months same house 1 year ago'].astype(str).str.replace(',', ''))
#had to change to str, remove commas and change to numeric
GeoMobilityIncomeDF['median income past 12 months moved within same county'] = pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months moved within same county'].astype(str).str.replace(',', ''))
#had to change to str, remove commas
GeoMobilityIncomeDF['median income past 12 months moved from different county same state'] = GeoMobilityIncomeDF['median income past 12 months moved from different county same state'].astype(str).str.replace(',', '')
#then remove hyphens and change to numeric
GeoMobilityIncomeDF['median income past 12 months moved from different county same state'] = pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months moved from different county same state'].astype(str).str.replace('-', ''))
#had to change to str, remove commas and change to numeric
GeoMobilityIncomeDF['median income past 12 months moved from different state'] = pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months moved from different state'].astype(str).str.replace(',', ''))
#had to change to str, remove commas
GeoMobilityIncomeDF['median income past 12 months moved from abroad'] = GeoMobilityIncomeDF['median income past 12 months moved from abroad'].astype(str).str.replace(',', '')
#then remove plus signs
GeoMobilityIncomeDF['median income past 12 months moved from abroad'] = GeoMobilityIncomeDF['median income past 12 months moved from abroad'].astype(str).str.replace('+', '')
#then remove hyphens and change to numeric
GeoMobilityIncomeDF['median income past 12 months moved from abroad'] = pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months moved from abroad'].astype(str).str.replace('-', ''))


#run basic stats on all columns
GeoMobilityIncomeDF.describe().applymap('{:,.0f}'.format)

In [ ]:
#calculate the percent change of median income past 12 months vs median income past 12 months for people that moved in from out of state

#create an empty column and insert the percent change calculation
#changes columns to numeric first
GeoMobilityIncomeDF['percent_change_median income out of state'] = (
    (pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months moved from different state']) -
     pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months'])) /
    pd.to_numeric(GeoMobilityIncomeDF['median income past 12 months'])
) * 100

#change the percent change column to integer type and round numbers with no decimals
GeoMobilityIncomeDF['percent_change_median income out of state'] = (
   GeoMobilityIncomeDF['percent_change_median income out of state'].round().astype(int))

GeoMobilityIncomeDF

In [ ]:
#sort the DF on the percent change column in descending order
GeoMobilityIncomeDF = GeoMobilityIncomeDF.sort_values(by=['percent_change_median income out of state'], ascending=False)

#print out the top 10 with most percent change
GeoMobilityIncomeDF.head(10)

In [ ]:
#create a new DF with our sorted percent change results and the columns used to calculate
new_GeoMobilityIncomeDF = GeoMobilityIncomeDF[['geo id', 'county name', 'median income past 12 months', 'median income past 12 months moved from different state', 'percent_change_median income out of state']]

#print out only the top ten with the most percent change
new_GeoMobilityIncomeDF.head(10)

In [ ]:
#first file to be uploaded

infile2 = 'ACSDT1Y2023.B07204-Data_V2.csv'

#creates empty list
GeoMobilityList = []

#open file to read
with open(infile2, 'r') as csvfile:
  CensusReader2 = csv.reader(csvfile,  dialect='excel', delimiter=',')


#begin for loop to read in data into list
  for line in CensusReader2:
    if line[0] == '' or line[6].startswith('null'):
      continue
    else:
      try:
        #creates the dictionary and entries
        GeoMobility = {}

        GeoMobility['geo id'] = line[0]
        GeoMobility['county name'] = line[1]
        GeoMobility['total'] = line[2]
        GeoMobility['same house 1 year ago'] = line[3]
        GeoMobility['different house 1 year ago'] = line[4]
        GeoMobility['different house 1 year ago same city or town'] = line[5]
        GeoMobility['different house 1 year ago same city or town same county'] = line[6]
        GeoMobility['different house 1 year ago same city or town different county same state'] = line[7]
        GeoMobility['different house 1 year ago elsewhere total'] = line[8]
        GeoMobility['different house 1 year ago elsewhere same county'] = line[9]
        GeoMobility['different house 1 year ago elsewhere different county'] = line[10]
        GeoMobility['different house 1 year ago elsewhere different county same state'] = line[11]
        GeoMobility['different house 1 year ago elsewhere different county different state'] = line[12]
        GeoMobility['different house 1 year ago elsewhere different state northeast'] = line[13]
        GeoMobility['different house 1 year ago elsewhere different state midwest'] = line[14]
        GeoMobility['different house 1 year ago elsewhere different state south'] = line[15]
        GeoMobility['different house 1 year ago elsewhere different state west'] = line[16]
        GeoMobility['abroad 1 year ago'] = line[17]
        GeoMobility['abroad 1 year ago puerto rico'] = line[18]
        GeoMobility['abroad 1 year ago US island areas'] = line[19]
        GeoMobility['abroad 1 year ago foreign country'] = line[20]



        #adds the dictionary entries into an account list
        GeoMobilityList.append(GeoMobility)

      except IndexError:
        print('Error: ', line)

csvfile.close()
#prints number of read entries
print('Read', len(GeoMobilityList), 'counties')

In [ ]:
#remove the top two rows

GeoMobilityList = GeoMobilityList[2:]

print(GeoMobilityList)

In [ ]:
#turn into a pandas DF

GeoMobilityDF = pd.DataFrame(GeoMobilityList)

GeoMobilityDF

In [ ]:
#switched all columns to numeric and integer type
GeoMobilityDF['total'] = pd.to_numeric(GeoMobilityDF['total'])
GeoMobilityDF['same house 1 year ago'] = pd.to_numeric(GeoMobilityDF['same house 1 year ago'])
GeoMobilityDF['different house 1 year ago'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago'])
GeoMobilityDF['different house 1 year ago same city or town'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago same city or town'])
GeoMobilityDF['different house 1 year ago same city or town same county'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago same city or town same county'])
GeoMobilityDF['different house 1 year ago same city or town different county same state'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago same city or town different county same state'])
GeoMobilityDF['different house 1 year ago elsewhere total'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere total'])
GeoMobilityDF['different house 1 year ago elsewhere same county'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere same county'])
GeoMobilityDF['different house 1 year ago elsewhere different county'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different county'])
GeoMobilityDF['different house 1 year ago elsewhere different county same state'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different county same state'])
GeoMobilityDF['different house 1 year ago elsewhere different county different state'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different county different state'])
GeoMobilityDF['different house 1 year ago elsewhere different state northeast'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state northeast'])
GeoMobilityDF['different house 1 year ago elsewhere different state midwest'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state midwest'])
GeoMobilityDF['different house 1 year ago elsewhere different state south'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state south'])
GeoMobilityDF['different house 1 year ago elsewhere different state west'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state west'])
GeoMobilityDF['abroad 1 year ago'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago'])
GeoMobilityDF['abroad 1 year ago puerto rico'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago puerto rico'])
GeoMobilityDF['abroad 1 year ago US island areas'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago US island areas'])
GeoMobilityDF['abroad 1 year ago foreign country'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago foreign country'])
GeoMobilityDF['different house 1 year ago elsewhere different state midwest'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state midwest'])
GeoMobilityDF['different house 1 year ago elsewhere different state south'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state south'])
GeoMobilityDF['different house 1 year ago elsewhere different state west'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different state west'])
GeoMobilityDF['abroad 1 year ago'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago'])
GeoMobilityDF['abroad 1 year ago puerto rico'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago puerto rico'])
GeoMobilityDF['abroad 1 year ago US island areas'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago US island areas'])
GeoMobilityDF['abroad 1 year ago foreign country'] = pd.to_numeric(GeoMobilityDF['abroad 1 year ago foreign country'])



#run basic stats on all columns
GeoMobilityDF.describe().applymap('{:,.0f}'.format)

In [ ]:
#checking the type
type(GeoMobilityDF['total'][0])

In [ ]:
GeoMobilityDF.to_csv('GeoMobilityDF.csv', index=False)

In [ ]:
#calculate people that moved in from out of state, this is people who lived in the same county name outside of the state, and people to lived in a different county in a different state

GeoMobilityDF['lived out of state'] = pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere same county']) + pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere different county different state'])

GeoMobilityDF

In [ ]:
#normalize the people moving in from out of state by dividing the number of people that lived out of state by the total numver of people that lived elsewhere

GeoMobilityDF['normalized_lived out of state'] = (
    GeoMobilityDF['lived out of state'] / pd.to_numeric(GeoMobilityDF['different house 1 year ago elsewhere total'])
)

GeoMobilityDF

In [ ]:
#sort in descending order by the normalized lived out of state

GeoMobilityDF = GeoMobilityDF.sort_values(by=['normalized_lived out of state'], ascending=False)

#print out the top 10 with the highest normalized percetange rate
GeoMobilityDF.head(10)

In [ ]:
#create a new DF with our sorted normalized results and the columns used to calculate

new_GeoMobilityDF = GeoMobilityDF[['geo id', 'county name', 'different house 1 year ago elsewhere total', 'lived out of state', 'normalized_lived out of state']]

new_GeoMobilityDF.head(10)

In [ ]:
#merge both of our new results DFs by the geo id column

merged_df = pd.merge(new_GeoMobilityDF, new_GeoMobilityIncomeDF, on='geo id', how='inner')

merged_df

In [ ]:
#here we do a 2 layer sort, first sorting by percent change median income and then by the normalized rate of people that lived out of state

merged_df = merged_df.sort_values(by=['percent_change_median income out of state','normalized_lived out of state'], ascending=[False, False])


merged_df.head(10)

In [ ]:
merged_df.to_csv('merged_df.csv', index=False)

In [ ]:
'''After we have all of our data analyzed, we want to create some intuitive maps to show our recommendations'''



In [ ]:
#load data file
gdf = gpd.read_file('Texas_acs2022_5yr_B07011_05000US48211.geojson')

In [ ]:
#get number of rows and columns
gdf.shape

In [ ]:
#drops first 3 rows
gdf = gdf.drop(range(0,3))

gdf

In [ ]:
#print a plot of the data
gdf.plot(figsize=(10,10))

In [ ]:
#plot a random row of data
gdf.sample().plot()

In [ ]:
#look into the data types in geodataframe
gdf.info()

In [ ]:
#print some of the DF
gdf.head()

In [ ]:
#view the columns in the df

list(gdf)

In [ ]:
#pick which columns to keep, keeping all of them this time

column_to_keep = ['geoid','name','B07011001','B07011002','B07011003', 'B07011004', 'B07011005', 'B07011006','geometry']

gdf = gdf[column_to_keep]

gdf

In [ ]:
#rename the columns

gdf.columns = ['geoid','name','median income past 12 months','same house 1 year ago', 'moved within same county', 'moved from a different county within the same state', 'moved from a different state', 'moved from abroad','geometry']

gdf

In [ ]:
#fill NaN with 0s

gdf = gdf.fillna(0)

gdf

In [ ]:
#plot a histogram of the median income of people moved from a different state

gdf['moved from a different state'].plot.hist(figsize=(10,10),
                                              bins=50,
                                              title='Texas, Median Income of People Moved from a Different State')

In [ ]:
#plot a heat map of the median income of people that moved from out of state
gdf.plot(figsize=(10,10),
                 column='moved from a different state',
                 cmap='plasma',
                 legend=True)

In [ ]:
#reproject to mercator for placing a base map
gdf_web_mercator = gdf.to_crs(epsg=3857)

gdf_web_mercator

In [ ]:
#define the plot
fig, ax = plt.subplots(figsize=(10,10))

#plot on people that moved from a different state and desired output
gdf_web_mercator[gdf_web_mercator['moved from a different state'] > 75000].plot(ax=ax, alpha=0.8)

#turn the axis off
ax.axis('off')

#title for the plot
ax.set_title('Texas Counties, Median Income of People Moved from a Different State more than $75,000', fontsize=15)

#add the basemap
ctx.add_basemap(ax)

In [ ]:
##first file to be uploaded

infile = 'GeoMobilityIncomeDF_results_FOR MAPS_SHORT.csv'

#creates empty list
ResultsList = []

#open file to read
with open(infile, 'r') as csvfile:
  ResultsReader = csv.reader(csvfile,  dialect='excel', delimiter=',')


#begin for loop to read in data into list
  for line in ResultsReader:
    if line[0] == '':
      continue
    else:
      try:
        #creates the dictionary and entries
        Results = {}

        Results['geoid'] = line[0]
        Results['county name'] = line[1]
        Results['percent_change_median income out of state'] = line[2]



        #adds the dictionary entries into an account list
        ResultsList.append(Results)

      except IndexError:
        print('Error: ', line)

csvfile.close()
#prints number of read entries
print('Read', len(ResultsList), 'counties')

In [ ]:
##
ResultsList = ResultsList[1:]

ResultsDF = pd.DataFrame(ResultsList)

ResultsDF

In [ ]:
columns_to_keep2 = ['geoid','geometry','name']

gdf2 = gdf[columns_to_keep2]

gdf2['geoid'] = gdf2['geoid'].astype(str)

gdf2


In [ ]:
##

ResultsDF['geoid'] = ResultsDF['geoid'].astype(str)

string = '05000'

#replace the first 7 characters of geoid with string
ResultsDF['geoid'] = ResultsDF['geoid'].str.replace(r'^.{7}', string, regex=True)

ResultsDF

In [ ]:
##

merged_results_gdf = gdf2.merge(ResultsDF, on='geoid', how='inner')

merged_results_gdf['percent_change_median income out of state'] = merged_results_gdf['percent_change_median income out of state'].astype(float)

merged_results_gdf

In [ ]:
##plot a heat map of the median income of people that moved from out of state
merged_results_gdf.plot(figsize=(10,10),
                 column='percent_change_median income out of state',
                 cmap='plasma',
                 legend=True)

In [ ]:
##reproject to mercator for placing a base map
merged_gdf_web_mercator = merged_results_gdf.to_crs(epsg=3857)

merged_gdf_web_mercator

In [ ]:
##define the plot
fig, ax = plt.subplots(figsize=(10,10))

#plot on people that moved from a different state and desired output
merged_gdf_web_mercator[merged_gdf_web_mercator['percent_change_median income out of state'] > 50].plot(ax=ax, alpha=0.8)

#turn the axis off
ax.axis('off')

#title for the plot
ax.set_title('Texas Counties, People Moving into Texas with 50% Greater Median Income', fontsize=15)

#add the basemap
ctx.add_basemap(ax)

In [ ]:
#first file to be uploaded

infile2 = '2023 Mobility and Income_Census Data_Final Results_FOR MAPS.csv'

#creates empty list
ResultsList2 = []

#open file to read
with open(infile2, 'r') as csvfile:
  ResultsReader2 = csv.reader(csvfile,  dialect='excel', delimiter=',')


#begin for loop to read in data into list
  for line in ResultsReader2:
    if line[0] == '':
      continue
    else:
      try:
        #creates the dictionary and entries
        Results2 = {}

        Results2['geoid'] = line[0]
        Results2['county name'] = line[1]
        Results2['different house 1 year ago elsewhere total'] = line[2]
        Results2['lived out of state'] = line[3]
        Results2['normalized lived out of state'] = line[4]
        Results2['median income past 12 months'] = line[6]
        Results2['median income past 12 months moved from different state'] = line[7]
        Results2['percent_change_median income out of state'] = line[8]



        #adds the dictionary entries into an account list
        ResultsList2.append(Results2)

      except IndexError:
        print('Error: ', line)

csvfile.close()
#prints number of read entries
print('Read', len(ResultsList2), 'counties')

In [ ]:
ResultsList2 = ResultsList2[1:]

ResultsDF2 = pd.DataFrame(ResultsList2)

ResultsDF2

In [ ]:
ResultsDF2['geoid'] = ResultsDF2['geoid'].astype(str)

string = '05000'

#replace the first 7 characters of geoid with string
ResultsDF2['geoid'] = ResultsDF2['geoid'].str.replace(r'^.{7}', string, regex=True)

ResultsDF2

In [ ]:
merged_results_gdf2 = gdf2.merge(ResultsDF2, on='geoid', how='inner')

merged_results_gdf2['percent_change_median income out of state'] = merged_results_gdf2['percent_change_median income out of state'].astype(float)

merged_results_gdf2

In [ ]:
#plot a heat map of the median income of people that moved from out of state
merged_results_gdf2.plot(figsize=(10,10),
                 column='percent_change_median income out of state',
                 cmap='plasma',
                 legend=True)

In [ ]:
#reproject to mercator for placing a base map
merged_gdf2_web_mercator = merged_results_gdf2.to_crs(epsg=3857)

merged_gdf2_web_mercator

In [ ]:
#define the plot
fig, ax = plt.subplots(figsize=(10,10))

#plot on people that moved from a different state and desired output
merged_gdf2_web_mercator[merged_gdf2_web_mercator['percent_change_median income out of state'] > 50].plot(ax=ax, alpha=0.8)

#turn the axis off
ax.axis('off')

#title for the plot
ax.set_title('Texas Counties, People Moving into Texas with 50% Greater Median Income', fontsize=15)

#add the basemap
ctx.add_basemap(ax)